# DDL / Spark SQL Views

Dieses Notebook lädt die bereinigten Parquet-Daten aus dem HDFS und registriert sie als Spark SQL View.

## Ziel

- processed Parquet-Daten lesen
- Schema prüfen
- Spark SQL View `parking_violations` erstellen
- erste SQL-Testqueries ausführen
- prüfen, ob die bereinigten Datumsfelder (`issue_month`, `issue_weekday`) für zeitliche Analysen verfügbar sind
- prüfen, ob die bereinigte Zeitspalte `violation_hour` für Tageszeit-Analysen verwendet werden kann

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_DDL") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 09:20:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

df.printSchema()

root
 |-- violation_code: string (nullable = true)
 |-- summons_number: string (nullable = true)
 |-- plate_id: string (nullable = true)
 |-- registration_state: string (nullable = true)
 |-- violation_time: string (nullable = true)
 |-- violation_county: string (nullable = true)
 |-- violation_precinct: integer (nullable = true)
 |-- street_name: string (nullable = true)
 |-- vehicle_make: string (nullable = true)
 |-- vehicle_body_type: string (nullable = true)
 |-- issue_date_parsed: date (nullable = true)
 |-- issue_year: integer (nullable = true)
 |-- issue_month: integer (nullable = true)
 |-- issue_weekday: integer (nullable = true)
 |-- violation_minute: integer (nullable = true)
 |-- violation_hour: integer (nullable = true)
 |-- violation_description_official: string (nullable = true)
 |-- fine_manhattan_96_below: long (nullable = true)
 |-- fine_other_areas: long (nullable = true)
 |-- fiscal_year: integer (nullable = true)



In [3]:
df.createOrReplaceTempView("parking_violations")

spark.sql("""
SELECT *
FROM parking_violations
LIMIT 5
""").show(truncate=False)

+--------------+--------------+--------+------------------+--------------+----------------+------------------+---------------+------------+-----------------+-----------------+----------+-----------+-------------+----------------+--------------+------------------------------+-----------------------+----------------+-----------+
|violation_code|summons_number|plate_id|registration_state|violation_time|violation_county|violation_precinct|street_name    |vehicle_make|vehicle_body_type|issue_date_parsed|issue_year|issue_month|issue_weekday|violation_minute|violation_hour|violation_description_official|fine_manhattan_96_below|fine_other_areas|fiscal_year|
+--------------+--------------+--------+------------------+--------------+----------------+------------------+---------------+------------+-----------------+-----------------+----------+-----------+-------------+----------------+--------------+------------------------------+-----------------------+----------------+-----------+
|14          

In [4]:
spark.sql("""
SELECT
    fiscal_year,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY fiscal_year
ORDER BY fiscal_year
""").show()

[Stage 2:================================================>        (16 + 3) / 19]

+-----------+--------------------+
|fiscal_year|number_of_violations|
+-----------+--------------------+
|       2023|            21562121|
|       2024|            11848421|
|       2025|            16557021|
+-----------+--------------------+



In [5]:
spark.sql("""
SELECT
    violation_code,
    violation_description_official,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY violation_code, violation_description_official
ORDER BY number_of_violations DESC
LIMIT 20
""").show(truncate=False)


[Stage 5:================================================>        (16 + 3) / 19]

+--------------+------------------------------+--------------------+
|violation_code|violation_description_official|number_of_violations|
+--------------+------------------------------+--------------------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|16853673            |
|21            |NO PARKING-STREET CLEANING    |5828637             |
|38            |FAIL TO DSPLY MUNI METER RECPT|3513550             |
|14            |NO STANDING-DAY/TIME LIMITS   |2516377             |
|5             |BUS LANE VIOLATION            |2150369             |
|7             |FAILURE TO STOP AT RED LIGHT  |2068262             |
|40            |FIRE HYDRANT                  |2016485             |
|20            |NO PARKING-DAY/TIME LIMITS    |1920515             |
|71            |INSP. STICKER-EXPIRED/MISSING |1757404             |
|70            |REG. STICKER-EXPIRED/MISSING  |1191802             |
|46            |DOUBLE PARKING                |1035778             |
|37            |EXPIRED MUNI METER

In [6]:
spark.sql("""
SELECT
    vehicle_make,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY vehicle_make
ORDER BY number_of_violations DESC
LIMIT 20
""").show(truncate=False)

[Stage 8:======================================================>  (18 + 1) / 19]

+------------+--------------------+
|vehicle_make|number_of_violations|
+------------+--------------------+
|HONDA       |5913518             |
|TOYOT       |5874992             |
|FORD        |4644984             |
|NISSA       |3906759             |
|CHEVR       |2665598             |
|ME/BE       |2583082             |
|BMW         |2466884             |
|JEEP        |2286474             |
|HYUND       |1719808             |
|LEXUS       |1237732             |
|FRUEH       |1121149             |
|ACURA       |1099422             |
|SUBAR       |1082377             |
|KIA         |1033208             |
|DODGE       |988788              |
|AUDI        |949137              |
|MAZDA       |920172              |
|VOLKS       |919156              |
|RAM         |759273              |
|INFIN       |746363              |
+------------+--------------------+



In [7]:
spark.sql("""
SELECT
    fiscal_year,
    issue_month,
    COUNT(*) AS number_of_violations
FROM parking_violations
GROUP BY fiscal_year, issue_month
ORDER BY fiscal_year, issue_month
""").show(50)

26/05/24 09:20:33 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 11:===================================>                    (12 + 4) / 19]

+-----------+-----------+--------------------+
|fiscal_year|issue_month|number_of_violations|
+-----------+-----------+--------------------+
|       2023|          1|             1347529|
|       2023|          2|             1280177|
|       2023|          3|             1520120|
|       2023|          4|             1388308|
|       2023|          5|             1514413|
|       2023|          6|             1652983|
|       2023|          7|             2863817|
|       2023|          8|             3256112|
|       2023|          9|             2506415|
|       2023|         10|             1520829|
|       2023|         11|             1453280|
|       2023|         12|             1258138|
|       2024|          1|             1216013|
|       2024|          2|             1251491|
|       2024|          3|             1350136|
|       2024|          4|             1305517|
|       2024|          5|             1433654|
|       2024|          6|             1082126|
|       2024|

In [8]:
spark.sql("""
SELECT
    violation_hour,
    COUNT(*) AS number_of_violations
FROM parking_violations
WHERE violation_hour IS NOT NULL
GROUP BY violation_hour
ORDER BY violation_hour
""").show(24)

[Stage 14:===================================>                    (12 + 4) / 19]

+--------------+--------------------+
|violation_hour|number_of_violations|
+--------------+--------------------+
|             0|              629283|
|             1|              729028|
|             2|              574919|
|             3|              472768|
|             4|              445412|
|             5|              698444|
|             6|             1526631|
|             7|             2797308|
|             8|             4225453|
|             9|             4391973|
|            10|             3591065|
|            11|             4367214|
|            12|             4084772|
|            13|             3880561|
|            14|             3552312|
|            15|             3002091|
|            16|             2393389|
|            17|             2059609|
|            18|             1526229|
|            19|             1127158|
|            20|             1116317|
|            21|              997842|
|            22|              875886|
|           

## Ergebnis

Die bereinigten Parquet-Daten konnten erfolgreich aus dem HDFS gelesen und als Spark SQL View `parking_violations` registriert werden.

Die Testqueries zeigen:

- Die Anzahl Zeilen pro Fiskaljahr stimmt mit dem Pre-processing überein.
- `violation_code = 36` ist mit Abstand der häufigste Violation Code.
- Die häufigsten Vehicle Makes sind unter anderem `HONDA`, `TOYOT`, `FORD` und `NISSA`.
- Die Monatsverteilung kann pro Fiskaljahr mit SQL abgefragt werden.
- Die bereinigte Spalte `violation_hour` kann für Tageszeit-Analysen verwendet werden.
- Für Tageszeit-Analysen sollen nur Zeilen mit `violation_hour IS NOT NULL` verwendet werden, da ungültige oder uneindeutige Zeitwerte im Pre-processing bewusst als `NULL` belassen wurden.
- `violation_description_official` liefert eine eindeutige offizielle Beschreibung pro `violation_code` aus der Mapping-Tabelle.


In [9]:
spark.stop()